[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_11_sharding_basics.ipynb)

# 🔴 Hard: Data-Parallel Mean with shard_map

*JAX Fundamentals*
Compute the mean of a `(B, D)` array **across devices**, using JAX's SPMD tools.

Split the batch dimension across all available devices, have each device reduce
its own shard, then combine with a collective. Return the scalar global mean.

### Rules
- Build a 1-D mesh over `jax.devices()` with axis name `"data"`
- Use `jax.shard_map` with `in_specs=P("data", None)` and `out_specs=P()`
- Combine the per-device results with `jax.lax.pmean`
- `B` is divisible by the device count
- The result must match `x.mean()` to float tolerance

### Explicit vs Auto axis types
Recent JAX versions give mesh axes a *type*. `jax.make_mesh(...)` defaults to
**Explicit**, which means `shard_map` insists the input already carries a
matching sharding and errors otherwise:

```
ValueError: in_specs passed to shard_map: P('data', None) does not match
the specs of the input: P(None, None)
```

Ways out: place the array yourself first with
`jax.device_put(x, NamedSharding(mesh, P("data", None)))`; use `jax.reshard(x,
P("data", None))` inside a `with jax.sharding.set_mesh(mesh):` block, which is
what the error message itself suggests; or declare the axis **Auto** and let
`shard_map` do the placement:

```python
mesh = jax.make_mesh((n,), ("data",), axis_types=(jax.sharding.AxisType.Auto,))
```

This task uses the `Auto` route, because it keeps `data_parallel_mean(x)` a
plain function of a plain array.

### Signature
```python
def data_parallel_mean(x):  # (B, D) -> scalar
    ...
```

### Running this without a TPU pod
On a single CPU you can fake 8 devices — this must be set **before** the first
`jax` import:

```python
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"
import jax
print(jax.devices())   # 8 CpuDevices
```

### Why it matters
Inside `shard_map` you write code from the perspective of a **single device**:
shapes are per-shard, and cross-device communication is explicit via collectives
(`pmean`, `psum`, `all_gather`, `ppermute`). That explicitness is the whole
point — it is why data parallelism, tensor parallelism, and FSDP are all just
different `PartitionSpec` choices over the same code.

The classic follow-up: *why is the mean-of-means correct here, and when does it
break?* (It breaks the moment shards have different row counts — then you need
`psum` of sums and `psum` of counts.)

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import os

# Fake 8 devices on a single CPU so the mesh has something to shard across.
# MUST run before jax is imported for the first time.
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P


def data_parallel_mean(x):
    """Global mean of x, computed with the batch sharded across devices.

    Args:
        x: (B, D) array, B divisible by len(jax.devices())

    Returns:
        Scalar mean, replicated across devices.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

print("devices:", len(jax.devices()))

x = jnp.arange(64.0).reshape(16, 4)
got = data_parallel_mean(x)
print("sharded mean:", got, " numpy mean:", x.mean())

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("sharding_basics")

# hint("sharding_basics")      # stuck? nudge without the answer
# solution("sharding_basics")  # spoiler: the reference implementation